In [209]:
import numpy as np
import torch
from template import *
import einops
from einops import *
import whisper

https://jessicastringham.net/2018/01/01/einsum/

Matrix Multiplication
np.einsum looks pretty scary. Matrix multiplication becomes:

np.einsum('ij,jk->ik', A, B)
ij,jk->ik, or the “einstein sum subscripts string,” tells einsum what it should do. The groups of letters are the operands and represent the arrays it should act on.

ij,jk->ik is defining a little function array1, array2 -> output.

Each letter labels an axis. ij is labeling the two axes of A.

I can read ij,jk->ik as “takes a 2D matrix, another 2D matrix, and returns a third 2D matrix.”

Then there are the rules:

-> repeating a letter on the left-hand side of the arrow means to multiply along those axes

-> omitting a letter from the right-hand side means sum over this axis.

-> the order of the letters in the output is the order of the array, so I can transpose too.

np.einsum is for reindexing, transposing, broadcasting, multiplying, summing/contracting, and taking diagonals.

In [ ]:
whisper_weights = torch.load('tiny.pt')

audio_input = np.load('/Users/timothyg/Documents/whisper_numpy/example_aud.npy')

In [252]:
whisper_weights['dims']['n_audio_state']

384

In [211]:
# print(whisper_weights['model_state_dict']['encoder.'])
print(audio_input.shape)

audio_input = np.einsum('bdt->btd', audio_input)
print(audio_input.shape)

(1, 80, 3000)
(1, 3000, 80)


# Convolution + GeLu


conv1: kernel_size=3, stride=1, padding=1
conv2: kernel_size=3, stride=2, padding=1


In [212]:
def gelu(x_arr): # docs.jax.dev/en/latest/_autosummary/jax.nn.gelu.html#jax.nn.gelu
    sqrt_2_over_pi = np.sqrt(2 / np.pi).astype(x_arr.dtype)
    cdf = 0.5 * (1.0 + np.tanh(sqrt_2_over_pi * (x_arr + 0.044715 * (x_arr ** 3))))
    return x_arr * cdf

# test_arr = np.array([-1,1,0,0.1,-0.1,111,-111])
# dbg(gelu(test_arr))

# m = torch.nn.GELU()
# output = m(torch.tensor(test_arr))
# dbg(output)

In [213]:
for k,v in whisper_weights['model_state_dict'].items():
    if 'encoder' in k and 'conv' in k:
        print(k)
        print(v.shape)

encoder.conv1.weight
torch.Size([384, 80, 3])
encoder.conv1.bias
torch.Size([384])
encoder.conv2.weight
torch.Size([384, 384, 3])
encoder.conv2.bias
torch.Size([384])


In [214]:
print(audio_input.shape)

(1, 3000, 80)


In [215]:
x = audio_input

for i in [1,2]:
    W, b = np.array(whisper_weights['model_state_dict'][f'encoder.conv{i}.weight']), np.array(whisper_weights['model_state_dict'][f'encoder.conv{i}.bias'])
    print(W.shape, b.shape)

    B, T, C = x.shape

    pad = np.zeros((B, 1, C))
    x, _ = pack([pad, x, pad], 'b * c')

    x = einsum(np.array([
        x[:, :-2:i, :],  # stride 2 for layer 2
        x[:, 1:-1:i, :],
        x[:, 2::i, :]

    ]), W, 'convdim b t u, v u convdim -> b t v') # contract feature dimension

    # x = reduce(x, 'b t convdim v -> b t v', 'sum') # convolution: sum in window

    x += b

    x = gelu(x)

(384, 80, 3) (384,)
(384, 384, 3) (384,)


/var/folders/c0/_s4rkd9926n5bcpmjnvx8ljr0000gn/T/ipykernel_88014/3641015628.py:4: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  W, b = np.array(whisper_weights['model_state_dict'][f'encoder.conv{i}.weight']), np.array(whisper_weights['model_state_dict'][f'encoder.conv{i}.bias'])


In [216]:
check_out = x

In [217]:
print(check_out)

[[[-0.08814428 -0.04524309  0.35468481 ... -0.13971014 -0.00283769
   -0.10262897]
  [-0.14510726  0.01255475  0.23418739 ... -0.15793346 -0.0045998
   -0.16133775]
  [-0.08617463  0.28288383  0.40084942 ... -0.1106639  -0.02969584
   -0.13048121]
  ...
  [ 0.00942291 -0.16035063  0.43869378 ... -0.16893373 -0.0140375
   -0.16730806]
  [-0.05129321 -0.14450865  0.20992646 ... -0.16920694 -0.00114394
   -0.16199679]
  [-0.04761317 -0.04585365  0.30849773 ... -0.16596946 -0.0067557
   -0.00606693]]]


In [218]:
import torch.nn.functional as F

sd = whisper_weights['model_state_dict']

with torch.no_grad():
    x_torch = torch.from_numpy(audio_input.astype(np.half)).permute(0, 2, 1)

    y = F.gelu(F.conv1d(x_torch, sd["encoder.conv1.weight"], sd["encoder.conv1.bias"], padding=1))
    
    print(y)

    y = F.gelu(F.conv1d(y, sd["encoder.conv2.weight"], sd["encoder.conv2.bias"], stride=2, padding=1))
    y = y.permute(0, 2, 1).cpu().numpy()

tensor([[[ 0.0432,  0.0344,  0.0166,  ...,  0.0942,  0.1298,  0.1655],
         [-0.0486, -0.0906, -0.1234,  ..., -0.0007, -0.0008, -0.0229],
         [-0.1583, -0.1331, -0.1411,  ..., -0.0475, -0.0644, -0.1107],
         ...,
         [-0.0010, -0.1119, -0.1655,  ..., -0.1410, -0.1627,  0.1941],
         [-0.1499, -0.1371, -0.0848,  ..., -0.1384, -0.1495, -0.1176],
         [ 0.1031,  0.0700,  0.0286,  ...,  0.1582,  0.1161,  0.1071]]],
       dtype=torch.float16)


In [219]:
print(np.max(np.abs(check_out - y)))

0.003833244933788116


In [220]:
temp_x = x

Positional Embedding

In [221]:
sd['encoder.positional_embedding'].shape

torch.Size([1500, 384])

In [222]:
x += np.array(sd['encoder.positional_embedding'])

/var/folders/c0/_s4rkd9926n5bcpmjnvx8ljr0000gn/T/ipykernel_88014/1216371676.py:1: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  x += np.array(sd['encoder.positional_embedding'])


In [223]:
x # this is the shape of the residual stream the whole way through

array([[[-0.08814428, -0.04524309,  0.35468481, ...,  0.86028986,
          0.99716231,  0.89737103],
        [ 0.69620133,  0.82749615,  1.02227333, ...,  0.84206654,
          0.9954002 ,  0.83866225],
        [ 0.82300505,  1.22721977,  1.37106426, ...,  0.8893361 ,
          0.97030416,  0.86951879],
        ...,
        [ 1.00893463,  0.07927339,  1.24728753, ...,  0.81739439,
          0.97375547,  0.82146147],
        [ 0.46237867,  0.78566713,  0.24364838, ...,  0.81712118,
          0.98664903,  0.82677274],
        [-0.4919491 ,  0.79203697, -0.45859212, ...,  0.82035866,
          0.98103726,  0.9827026 ]]], shape=(1, 1500, 384))

Attention: self attention

In [224]:
def ln(x, W, b):
    x = x - np.mean(x, axis=-1, keepdims=True)
    x = x / (np.std(x, axis=-1, keepdims=True) + 1e-9)
    return W * x + b


print(ln(x, np.array(sd['encoder.blocks.0.attn_ln.weight']), np.array(sd['encoder.ln_post.bias'])))

[[[-1.98422924 -1.6507969  -0.79041669 ...  1.35538363  3.24055191
    1.27606294]
  [-0.41957536  0.12024688  0.36684574 ...  1.24900278  3.41972704
    1.08338998]
  [-0.15780993  0.99063435  0.98608897 ...  1.4687915   2.93907723
    1.1484612 ]
  ...
  [ 0.82983557 -0.66048448  1.14684123 ...  2.82693045  6.08385092
    1.99659881]
  [-0.15763517  0.67613932 -0.36309177 ...  2.89386058  6.32336322
    2.05054875]
  [-1.85558746  0.74777699 -1.37772668 ...  3.08042848  6.58748232
    2.60309164]]]


/var/folders/c0/_s4rkd9926n5bcpmjnvx8ljr0000gn/T/ipykernel_88014/1925866790.py:7: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  print(ln(x, np.array(sd['encoder.blocks.0.attn_ln.weight']), np.array(sd['encoder.ln_post.bias'])))


In [225]:
print(whisper_weights['dims'])

{'n_mels': 80, 'n_vocab': 51864, 'n_audio_ctx': 1500, 'n_audio_state': 384, 'n_audio_head': 6, 'n_audio_layer': 4, 'n_text_ctx': 448, 'n_text_state': 384, 'n_text_head': 6, 'n_text_layer': 4}


In [226]:
B, T, C = x.shape
n_heads = whisper_weights['dims']['n_audio_head']
n_layers = whisper_weights['dims']['n_audio_layer']

for i in range(n_layers):

    # ==== self attention ====

    resid_x = x.copy()

    W_q, W_k, W_v, W_o = [np.array(sd[f'encoder.blocks.{i}.attn.{item}.weight']) for item in ['query', 'key', 'value', 'out']]
    # print(W_q.shape, W_k.shape, W_v.shape, W_o.shape)
    B_q, B_v, B_o = [np.array(sd[f'encoder.blocks.{i}.attn.{item}.bias']) for item in ['query', 'value', 'out']] 
    # softmax invariant to adding / substracting constant, no need for key bias!
    ln_w, ln_b = np.array([sd[f'encoder.blocks.{i}.attn_ln.weight'], sd[f'encoder.blocks.{i}.attn_ln.bias']])
    # print(ln_w.shape, ln_b.shape)
    

    x = ln(x, ln_w, ln_b)

    K, Q, V = x @ W_k.T, x @ W_q.T + B_q, x @ W_v.T + B_v

    K, Q, V = rearrange([K, Q, V], 'm batch time (n_heads feature) -> m batch n_heads time feature', n_heads=n_heads)

    scores = einsum(Q, K, 'b n t f, b n tt f -> b n t tt')

    scores /= np.sqrt(C / n_heads)

    scores = scores - np.max(scores, axis=-1, keepdims=True) # contracting dimension is tt
    scores = np.exp(scores)
    scores /= np.sum(scores, axis=-1, keepdims=True)

    # print(scores.shape)

    x = einsum(scores, V, 'b n t tt, b n tt f -> b n t f') # contracting row (post softmax scores)

    x = rearrange(x, 'b n t f -> b t (n f)')

    x = x @ W_o.T + B_o

    x += resid_x

    print(x)

    # ==== MLP ====

    resid_x = x.copy()

    W_1, W_2 = [np.array(sd[f'encoder.blocks.{i}.mlp.{item}.weight']) for item in ['0', '2']]
    B_1, B_2 = [np.array(sd[f'encoder.blocks.{i}.mlp.{item}.bias']) for item in ['0', '2']] 
    ln_w, ln_b = np.array([sd[f'encoder.blocks.{i}.mlp_ln.weight'], sd[f'encoder.blocks.{i}.mlp_ln.bias']])

    x = ln(x, ln_w, ln_b)

    x = x @ W_1.T
    x += B_1 # up proj

    x = gelu(x)

    x = x @ W_2.T
    x += B_2

    x += resid_x


/var/folders/c0/_s4rkd9926n5bcpmjnvx8ljr0000gn/T/ipykernel_88014/2947390533.py:11: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  W_q, W_k, W_v, W_o = [np.array(sd[f'encoder.blocks.{i}.attn.{item}.weight']) for item in ['query', 'key', 'value', 'out']]
/var/folders/c0/_s4rkd9926n5bcpmjnvx8ljr0000gn/T/ipykernel_88014/2947390533.py:13: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  B_q, B_v, B_o = [np.array(sd[f'encoder.blocks.{i}.attn.{item}.bias']) for item in ['query', 'valu

[[[ 0.11623385  0.57457753  0.37069831 ...  0.42041636  0.85635764
    0.65614255]
  [ 0.24344464  0.84354333  0.21620237 ...  0.23723292  0.79328465
    0.59059415]
  [ 0.4396064   0.85582849  0.33209038 ...  0.19392769  0.7442278
    0.62760356]
  ...
  [ 0.6464094   0.1922778   0.47142029 ... -0.02191418  0.99690001
    0.35423323]
  [ 0.64753797  0.29536003 -0.28265595 ...  0.00598747  0.93018718
    0.35071025]
  [ 0.08326308  1.21524534 -0.57645698 ...  0.03826741  0.92132698
    0.47444474]]]


/var/folders/c0/_s4rkd9926n5bcpmjnvx8ljr0000gn/T/ipykernel_88014/2947390533.py:49: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  W_1, W_2 = [np.array(sd[f'encoder.blocks.{i}.mlp.{item}.weight']) for item in ['0', '2']]
/var/folders/c0/_s4rkd9926n5bcpmjnvx8ljr0000gn/T/ipykernel_88014/2947390533.py:50: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  B_1, B_2 = [np.array(sd[f'encoder.blocks.{i}.mlp.{item}.bias']) for item in ['0', '2']]


[[[ 1.73048656e-01  4.75966700e-01  6.71938281e-01 ...  5.45187445e-01
    1.01226683e+00  3.95505868e-01]
  [ 1.97572676e-01  6.30061026e-01  3.65135904e-01 ...  3.52451543e-01
    1.00592521e+00  3.38491461e-01]
  [ 4.22914921e-01  5.15307389e-01  3.74804470e-01 ...  2.37494892e-01
    8.16969713e-01  5.99466886e-01]
  ...
  [ 9.81155173e-01  2.09351423e-01  8.81170953e-01 ...  5.19611235e-01
    6.78293835e-01  2.90779244e-01]
  [ 1.23199186e+00  6.24825563e-04 -4.27122677e-01 ...  6.22437462e-01
    7.87870545e-01  3.37288034e-01]
  [ 4.33948191e-01  1.30132999e+00 -3.74221092e-01 ...  5.27539760e-01
    7.23366897e-01  3.28816542e-01]]]
[[[ 0.2968235   0.00746111  0.81756591 ...  0.56782979  0.87762252
    0.62934195]
  [ 0.02449056  0.4201449   0.89242989 ...  0.44014998  0.90044952
    0.5212347 ]
  [ 0.44707319  0.34084268  0.40407699 ...  0.35657507  0.72282623
    0.6876915 ]
  ...
  [ 1.10952988  0.77393981  1.25846338 ...  0.54426789  0.28522918
    0.72592878]
  [ 1.690061

In [227]:
ln_w, ln_b = np.array([sd['encoder.ln_post.weight'], sd['encoder.ln_post.bias']])

x = ln(x, ln_w, ln_b)

# Testing

In [228]:
# === Correct final diff: your current x is already post encoder.ln_post ===

import numpy as np
import torch
import torch.nn.functional as F
from whisper.model import ModelDimensions, Whisper

try:
    from whisper.model import disable_sdpa
except ImportError:
    from contextlib import nullcontext
    def disable_sdpa():
        return nullcontext()

sd = whisper_weights["model_state_dict"]
dims = ModelDimensions(**whisper_weights["dims"])

def torch_to_np(t):
    return t.detach().cpu().float().numpy()

def report_diff(name, a, b):
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)

    if a.shape != b.shape:
        print(f"\n{name}")
        print(f"  SHAPE MISMATCH: numpy={a.shape}, torch={b.shape}")
        return

    d = a - b
    abs_d = np.abs(d)
    idx = np.unravel_index(abs_d.argmax(), abs_d.shape)

    print(f"\n{name}")
    print(f"  shape:    {a.shape}")
    print(f"  max_abs:  {abs_d.max():.8g}")
    print(f"  mean_abs: {abs_d.mean():.8g}")
    print(f"  rmse:     {np.sqrt(np.mean(d.astype(np.float64) ** 2)):.8g}")
    print(f"  worst idx {idx}: numpy={a[idx]:.8g}, torch={b[idx]:.8g}, diff={d[idx]:.8g}")

# PyTorch encoder expects mel as (batch, n_mels, time).
audio_np = np.asarray(audio_input)

if audio_np.shape[1] == dims.n_mels:
    mel_bdt = audio_np
elif audio_np.shape[2] == dims.n_mels:
    mel_bdt = np.transpose(audio_np, (0, 2, 1))
else:
    raise ValueError(f"Cannot infer mel axis from shape {audio_np.shape}")

mel_t = torch.from_numpy(mel_bdt.astype(np.float32, copy=False))

model = Whisper(dims).eval()
model.load_state_dict(sd, strict=False)
model = model.float()

with torch.no_grad(), disable_sdpa():
    torch_encoder_out = model.encoder(mel_t)

torch_encoder_out_np = torch_to_np(torch_encoder_out)

# Your current x should ALREADY include encoder.ln_post.
np_encoder_out = np.asarray(x, dtype=np.float32)

report_diff(
    "FINAL: your NumPy encoder output vs PyTorch encoder output",
    np_encoder_out,
    torch_encoder_out_np,
)


FINAL: your NumPy encoder output vs PyTorch encoder output
  shape:    (1, 1500, 384)
  max_abs:  0.063427389
  mean_abs: 0.0019299033
  rmse:     0.0025451272
  worst idx (np.int64(0), np.int64(10), np.int64(142)): numpy=-0.33270115, torch=-0.26927376, diff=-0.063427389


# Decoder

In [229]:
whisper_weights = torch.load('tiny.pt')
text_input = np.load('/Users/timothyg/Documents/whisper_numpy/example_toks.npy')

In [230]:
sd = whisper_weights['model_state_dict']

In [231]:
sd['decoder.positional_embedding'].shape

torch.Size([448, 384])

In [232]:
sd['decoder.token_embedding.weight'].shape

torch.Size([51864, 384])

In [233]:
vocab = np.array(sd['decoder.token_embedding.weight'])
input = vocab[text_input]
input.shape

/var/folders/c0/_s4rkd9926n5bcpmjnvx8ljr0000gn/T/ipykernel_88014/2427309955.py:1: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  vocab = np.array(sd['decoder.token_embedding.weight'])


(1, 6, 384)

In [234]:
sd['decoder.positional_embedding'].shape

torch.Size([448, 384])

In [235]:
B, T, C = input.shape

pos_embed = np.array(sd['decoder.positional_embedding'])[:T, :]

input += pos_embed

/var/folders/c0/_s4rkd9926n5bcpmjnvx8ljr0000gn/T/ipykernel_88014/3232959078.py:3: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  pos_embed = np.array(sd['decoder.positional_embedding'])[:T, :]


In [236]:
dbg(whisper_weights['dims']['n_text_ctx']) # max text toks, = pos_emd.shape[0]
dbg(whisper_weights['dims']['n_text_head']) # num heads for attention
dbg(whisper_weights['dims']['n_text_layer']) # num layers
dbg(whisper_weights['dims']['n_text_state']) # feature dim

2664.943s |> whisper_weights['dims']['n_text_ctx']: 448
2664.944s |> whisper_weights['dims']['n_text_head']: 6
2664.944s |> whisper_weights['dims']['n_text_layer']: 4
2664.944s |> whisper_weights['dims']['n_text_state']: 384


384

In [237]:
encoder_input = x.copy()

In [238]:
encoder_input.shape

(1, 1500, 384)

In [239]:
print(encoder_input)

[[[-0.66152912 -0.36595174  0.34210488 ... -1.20104859  0.41847386
   -0.32934625]
  [-1.13269586  0.75690895  0.75599923 ... -1.29893722 -0.11996806
   -0.18650404]
  [-0.77720709  0.08760896  0.47417431 ... -1.6930297  -0.28548497
    0.31262278]
  ...
  [ 0.53546757 -0.02444599  0.26032902 ... -0.42796165  0.1999561
   -0.24608828]
  [ 0.66201854  0.24204981 -1.50493876 ...  0.3258859   0.91765485
   -0.51260409]
  [-0.20884512  0.30016554 -0.51156938 ...  1.09488697 -0.27347068
    0.89238933]]]


In [240]:
x = input
x.shape

(1, 6, 384)

In [241]:
B, T, C = x.shape

assert T < whisper_weights['dims']['n_text_state']

n_heads = whisper_weights['dims']['n_text_head']
n_layers = whisper_weights['dims']['n_text_layer']

for i in range(n_layers):

    # ==== self attention ====

    resid_x = x.copy()

    W_q, W_k, W_v, W_o = [np.array(sd[f'decoder.blocks.{i}.attn.{item}.weight']) for item in ['query', 'key', 'value', 'out']]
    # print(W_q.shape, W_k.shape, W_v.shape, W_o.shape)
    B_q, B_v, B_o = [np.array(sd[f'decoder.blocks.{i}.attn.{item}.bias']) for item in ['query', 'value', 'out']] 
    # softmax invariant to adding / substracting constant, no need for key bias!
    ln_w, ln_b = np.array([sd[f'decoder.blocks.{i}.attn_ln.weight'], sd[f'decoder.blocks.{i}.attn_ln.bias']])
    # print(ln_w.shape, ln_b.shape)

    x = ln(x, ln_w, ln_b)

    K, Q, V = x @ W_k.T, x @ W_q.T + B_q, x @ W_v.T + B_v

    K, Q, V = rearrange([K, Q, V], 'm batch time (n_heads feature) -> m batch n_heads time feature', n_heads=n_heads)

    scores = einsum(Q, K, 'b n t f, b n tt f -> b n t tt')

    scores /= np.sqrt(C / n_heads)

    scores = scores - np.max(scores, axis=-1, keepdims=True) # contracting dimension is tt

    # Casual self-attention!

    mask = np.triu(np.full(scores.shape, -np.inf), k=1)
    scores = scores + mask
    scores = np.exp(scores)
    scores /= np.sum(scores, axis=-1, keepdims=True)

    # print(scores)

    # print(scores.shape)

    x = einsum(scores, V, 'b n t tt, b n tt f -> b n t f') # contracting row (post softmax scores)

    x = rearrange(x, 'b n t f -> b t (n f)')

    x = x @ W_o.T + B_o

    x += resid_x

    # ==== Cross attention ====

    resid_x = x.copy()

    W_q, W_k, W_v, W_o = [np.array(sd[f'decoder.blocks.{i}.cross_attn.{item}.weight']) for item in ['query', 'key', 'value', 'out']]
    B_q, B_v, B_o = [np.array(sd[f'decoder.blocks.{i}.cross_attn.{item}.bias']) for item in ['query', 'value', 'out']] 
    ln_w, ln_b = np.array([sd[f'decoder.blocks.{i}.cross_attn_ln.weight'], sd[f'decoder.blocks.{i}.cross_attn_ln.bias']])

    print("HERE HERE HERE", W_q.shape)

    x = ln(x, ln_w, ln_b)

    K, Q, V = encoder_input @ W_k.T, x @ W_q.T + B_q, encoder_input @ W_v.T + B_v

    K, V = rearrange([K, V], 'm b t (n_heads c) -> m b n_heads t c', n_heads=n_heads)
    Q = rearrange(Q, 'b tt (n_heads c) -> b n_heads tt c', n_heads=n_heads)

    scores = einsum(Q, K, 'b n tt c, b n t c -> b n tt t') # contract feature dimension

    scores /= np.sqrt(C / n_heads)

    scores = scores - np.max(scores, axis=-1, keepdims=True)
    scores = np.exp(scores)
    scores /= np.sum(scores, axis=-1, keepdims=True)

    print(scores.shape, V.shape)
    x = einsum(scores, V, 'b n tt t, b n t c -> b n tt c')

    print(x.shape)
    x = rearrange(x, 'b n tt c -> b tt (n c)')
    print(x.shape)

    x = x @ W_o.T + B_o

    x += resid_x

    # ==== MLP ====

    resid_x = x.copy()

    W_1, W_2 = [np.array(sd[f'decoder.blocks.{i}.mlp.{item}.weight']) for item in ['0', '2']]
    B_1, B_2 = [np.array(sd[f'decoder.blocks.{i}.mlp.{item}.bias']) for item in ['0', '2']] 
    ln_w, ln_b = np.array([sd[f'decoder.blocks.{i}.mlp_ln.weight'], sd[f'decoder.blocks.{i}.mlp_ln.bias']])

    x = ln(x, ln_w, ln_b)

    x = x @ W_1.T
    x += B_1 # up proj

    x = gelu(x)

    x = x @ W_2.T
    x += B_2

    x += resid_x


HERE HERE HERE (384, 384)
(1, 6, 6, 1500) (1, 6, 1500, 64)
(1, 6, 6, 64)
(1, 6, 384)
HERE HERE HERE (384, 384)
(1, 6, 6, 1500) (1, 6, 1500, 64)
(1, 6, 6, 64)
(1, 6, 384)
HERE HERE HERE (384, 384)
(1, 6, 6, 1500) (1, 6, 1500, 64)
(1, 6, 6, 64)
(1, 6, 384)
HERE HERE HERE (384, 384)
(1, 6, 6, 1500) (1, 6, 1500, 64)
(1, 6, 6, 64)
(1, 6, 384)


/var/folders/c0/_s4rkd9926n5bcpmjnvx8ljr0000gn/T/ipykernel_88014/567642458.py:14: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  W_q, W_k, W_v, W_o = [np.array(sd[f'decoder.blocks.{i}.attn.{item}.weight']) for item in ['query', 'key', 'value', 'out']]
/var/folders/c0/_s4rkd9926n5bcpmjnvx8ljr0000gn/T/ipykernel_88014/567642458.py:16: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  B_q, B_v, B_o = [np.array(sd[f'decoder.blocks.{i}.attn.{item}.bias']) for item in ['query', 'value'

In [242]:
ln_w, ln_b = np.array([sd['decoder.ln.weight'], sd['decoder.ln.bias']])

x = ln(x, ln_w, ln_b)

In [243]:
res = x @ np.array(sd['decoder.token_embedding.weight']).T

/var/folders/c0/_s4rkd9926n5bcpmjnvx8ljr0000gn/T/ipykernel_88014/3750489084.py:1: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  res = x @ np.array(sd['decoder.token_embedding.weight']).T


In [244]:
# res[0][-1].sort()
# res[0][-1]

In [245]:
# greedy decoding
res = np.argmax(res, axis=-1)

In [246]:
res

array([[50363,   383,  1310, 19490,   484,  1560]])

In [247]:
text_input

array([[50257, 50362,   383,  1310, 19490,   484]])

In [248]:
import numpy as np
import whisper

# Load token IDs
tokens = res

# Create tokenizer
tokenizer = whisper.tokenizer.get_tokenizer(multilingual=False)

# Remove special tokens if desired
# # (optional but usually helpful)
# special_tokens = set(tokenizer.special_tokens.values())

decoded = []
for seq in tokens:
    # filtered = [t for t in seq if t not in special_tokens]
    filtered = seq
    text = tokenizer.decode(filtered)
    decoded.append(text)

print(decoded)

[' The little tales they tell']


In [249]:
nxt, _ = pack([text_input, res[:, -1:]], 'b *')


In [250]:
nxt

array([[50257, 50362,   383,  1310, 19490,   484,  1560]])

In [251]:
np.save('/Users/timothyg/Documents/whisper_numpy/example_toks.npy', nxt)